# Phase 2: Monte Carlo Speech Simulation & Term Probability Engine

Loads the fine-tuned Trump LLM, runs 1,000 simulated speeches for an upcoming event,
extracts n-grams, and produces probability rankings for each tracked term.

**Input**: Fine-tuned LoRA adapter + event context + term list

**Output**: `predictions.json` with probability scores for each Kalshi market term

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'predictions')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Fine-Tuned Model

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B'
ADAPTER_PATH = os.path.join(MODEL_DIR, 'trump_lora_adapter')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,  # loads adapter on top of base
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 2. Configure the Event Context (RAG Input)

Edit this cell for each upcoming event you want to predict.

In [ ]:
import json

# ===== EDIT THIS FOR EACH EVENT =====
EVENT_CONTEXT = {
    'event_type': 'rally',
    'location': 'Michigan',
    'audience': 'auto workers and supporters',
    'date': '2026-03-15',
    'current_news': [
        'Trade tensions with China escalating over tariffs',
        'Auto industry layoffs in the Midwest',
        'Border security bill stalled in Congress',
        'Stock market hit new highs this week',
    ],
    'recent_trump_focus': [
        'Tariffs on Chinese goods',
        'Bringing manufacturing jobs back',
        'Border wall funding',
    ],
}

# Load historical context from your bot's export
TERM_CONTEXT_PATH = os.path.join(DATA_DIR, 'term_context.json')
EVENT_HISTORY_PATH = os.path.join(DATA_DIR, 'event_speech_pairs.json')

historical_context = ''
if os.path.exists(TERM_CONTEXT_PATH):
    with open(TERM_CONTEXT_PATH) as f:
        term_data = json.load(f)
    # Pull relevant historical quotes
    relevant_quotes = []
    for term in term_data:
        for ctx in term.get('contexts', [])[:3]:  # top 3 per term
            relevant_quotes.append(ctx['snippet'])
    historical_context = '\n'.join(relevant_quotes[:30])  # cap at 30

print(f'Event: {EVENT_CONTEXT["event_type"]} in {EVENT_CONTEXT["location"]}')
print(f'Historical quotes loaded: {len(historical_context.split(chr(10)))}')

In [ ]:
# Load the term list from your bot (the words Kalshi markets are tracking)
TERMS_PATH = os.path.join(DATA_DIR, 'term_context.json')

tracked_terms = []
if os.path.exists(TERMS_PATH):
    with open(TERMS_PATH) as f:
        term_data = json.load(f)
    tracked_terms = [t['normalized'] for t in term_data]
    print(f'Tracking {len(tracked_terms)} terms from Kalshi markets:')
    for t in tracked_terms[:20]:
        print(f'  - {t}')
    if len(tracked_terms) > 20:
        print(f'  ... and {len(tracked_terms) - 20} more')
else:
    print('No term list found. Upload term_context.json to', DATA_DIR)
    # Fallback: common Trump terms
    tracked_terms = [
        'tariff', 'china', 'border', 'wall', 'fake news', 'tremendous',
        'incredible', 'disaster', 'winning', 'great', 'beautiful',
        'billions', 'millions', 'radical left', 'deep state',
    ]

## 3. Build the RAG Prompt

In [ ]:
def build_generation_prompt(event_ctx: dict, hist_context: str = '') -> str:
    """Build the prompt that conditions the fine-tuned model on the event context.
    
    Since the model is already fine-tuned on Trump's speech patterns,
    we just need to seed it with the right context.
    """
    news_section = '\n'.join(f'- {n}' for n in event_ctx.get('current_news', []))
    focus_section = '\n'.join(f'- {f}' for f in event_ctx.get('recent_trump_focus', []))
    
    # The prompt seeds the generation with context
    # The fine-tuned model will continue in Trump's speech style
    prompt = f"""Thank you. Thank you very much, {event_ctx.get('location', 'everybody')}. \
What a crowd. We love {event_ctx.get('location', 'this place')}, don't we? \
We love the {event_ctx.get('audience', 'people')}. \
And I have to tell you, there are some very important things happening right now. \
"""
    
    return prompt

# Build variations of prompts for diverse generations
PROMPT_SEEDS = [
    build_generation_prompt(EVENT_CONTEXT, historical_context),
    f"Thank you, {EVENT_CONTEXT['location']}. We're going to talk about something very important today. ",
    f"Hello {EVENT_CONTEXT['location']}! What a beautiful place. Now, let me tell you about ",
    f"We love the {EVENT_CONTEXT.get('audience', 'people')} of {EVENT_CONTEXT['location']}. And you know what, ",
    f"This is a big day. A very big day for {EVENT_CONTEXT['location']}. Because we are going to ",
]

print(f'Created {len(PROMPT_SEEDS)} prompt variations')
print(f'\nSample prompt:\n{PROMPT_SEEDS[0][:300]}...')

## 4. Monte Carlo Simulation: Generate 1,000 Speeches

In [ ]:
from tqdm.notebook import tqdm
import random

NUM_SIMULATIONS = 1000
MAX_NEW_TOKENS = 512       # ~400 words per simulation
TEMPERATURE = 0.75         # slight variation while keeping core patterns
TOP_P = 0.9                # nucleus sampling
BATCH_SIZE = 10            # generate 10 at a time for GPU efficiency

all_speeches = []

print(f'Generating {NUM_SIMULATIONS} simulated speeches...')
print(f'Temperature: {TEMPERATURE} | Max tokens: {MAX_NEW_TOKENS}')
print(f'Estimated time: ~{NUM_SIMULATIONS * 2 / 60:.0f} minutes on A100\n')

for batch_start in tqdm(range(0, NUM_SIMULATIONS, BATCH_SIZE)):
    batch_prompts = []
    for i in range(batch_start, min(batch_start + BATCH_SIZE, NUM_SIMULATIONS)):
        # Rotate through prompt seeds for diversity
        prompt = PROMPT_SEEDS[i % len(PROMPT_SEEDS)]
        batch_prompts.append(prompt)
    
    # Tokenize batch
    inputs = tokenizer(
        batch_prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to('cuda')
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    # Decode
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        all_speeches.append(text)

print(f'\nGenerated {len(all_speeches)} speeches')
print(f'Average length: {sum(len(s.split()) for s in all_speeches) / len(all_speeches):.0f} words')

In [ ]:
# Preview a few samples
for i in [0, 500, 999]:
    if i < len(all_speeches):
        print(f'\n=== Speech #{i} ===')
        print(all_speeches[i][:500])
        print('...')

## 5. N-Gram Extraction & Term Probability Ranking

In [ ]:
import re
from collections import Counter

def extract_ngrams(text: str, n_range=(2, 5)) -> list[str]:
    """Extract n-grams from text."""
    words = re.findall(r'\b\w+\b', text.lower())
    ngrams = []
    for n in range(n_range[0], n_range[1] + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i + n]))
    return ngrams

def check_term_in_speech(speech: str, term: str) -> bool:
    """Check if a term (word or phrase) appears in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return bool(re.search(pattern, speech.lower()))

def count_term_in_speech(speech: str, term: str) -> int:
    """Count occurrences of a term in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return len(re.findall(pattern, speech.lower()))

print('Extraction functions ready.')

In [ ]:
# Calculate probability for each tracked Kalshi term

term_results = {}
total_speeches = len(all_speeches)

print(f'Analyzing {len(tracked_terms)} tracked terms across {total_speeches} simulated speeches...\n')

for term in tqdm(tracked_terms):
    # For compound terms (X / Y), check each sub-term
    sub_terms = [t.strip() for t in term.split('/')] if '/' in term else [term]
    
    speeches_containing = 0
    total_mentions = 0
    
    for speech in all_speeches:
        found = False
        for st in sub_terms:
            count = count_term_in_speech(speech, st)
            if count > 0:
                found = True
                total_mentions += count
        if found:
            speeches_containing += 1
    
    probability = speeches_containing / total_speeches
    avg_mentions = total_mentions / total_speeches
    
    term_results[term] = {
        'probability': round(probability, 4),
        'speeches_containing': speeches_containing,
        'total_mentions': total_mentions,
        'avg_mentions_per_speech': round(avg_mentions, 2),
        'confidence': round(min(1.0, total_speeches / 1000), 2),  # higher N = higher confidence
    }

# Sort by probability
sorted_terms = sorted(term_results.items(), key=lambda x: x[1]['probability'], reverse=True)

print(f'\n{"="*60}')
print(f'{"TERM PROBABILITY RANKINGS":^60}')
print(f'{"="*60}')
print(f'{"Term":<30} {"Probability":>12} {"Avg Mentions":>14}')
print('-' * 60)
for term, data in sorted_terms:
    prob_pct = f"{data['probability']:.1%}"
    avg = f"{data['avg_mentions_per_speech']:.1f}"
    print(f'{term:<30} {prob_pct:>12} {avg:>14}')

In [ ]:
# Also find the MOST COMMON N-GRAMS across all simulations
# (discovers phrases not in the Kalshi term list that Trump might say)

print('Extracting all n-grams from simulated speeches...')

all_ngrams = Counter()
for speech in tqdm(all_speeches):
    ngrams = extract_ngrams(speech, n_range=(2, 5))
    all_ngrams.update(ngrams)

# Filter out boring n-grams
stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'to', 'of',
             'in', 'for', 'on', 'with', 'at', 'by', 'and', 'or', 'but', 'not',
             'that', 'this', 'it', 'we', 'they', 'you', 'he', 'she', 'i',
             'do', 'did', 'does', 'have', 'has', 'had', 'will', 'would'}

interesting_ngrams = {}
for ngram, count in all_ngrams.most_common(500):
    words = ngram.split()
    # Skip if all words are stopwords
    if all(w in stopwords for w in words):
        continue
    # Must appear in at least 5% of speeches
    if count / total_speeches >= 0.05:
        interesting_ngrams[ngram] = {
            'count': count,
            'probability': round(count / total_speeches, 4),
        }

print(f'\n{"="*60}')
print(f'{"TOP PREDICTED PHRASES (not in Kalshi list)":^60}')
print(f'{"="*60}')

# Show n-grams NOT already in the tracked terms list
for ngram, data in sorted(interesting_ngrams.items(), key=lambda x: x[1]['count'], reverse=True)[:30]:
    if ngram not in tracked_terms:
        print(f"{data['probability']:.1%} - \"{ngram}\" ({data['count']} total mentions)")

## 6. Export Predictions for Your Trading Bot

In [ ]:
from datetime import datetime

# Build the final predictions output
predictions_output = {
    'generated_at': datetime.utcnow().isoformat(),
    'event_context': EVENT_CONTEXT,
    'simulation_params': {
        'num_simulations': NUM_SIMULATIONS,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_tokens': MAX_NEW_TOKENS,
        'base_model': BASE_MODEL,
    },
    'term_predictions': [
        {
            'term': term,
            'probability': data['probability'],
            'confidence': data['confidence'],
            'speeches_containing': data['speeches_containing'],
            'total_mentions': data['total_mentions'],
            'avg_mentions_per_speech': data['avg_mentions_per_speech'],
            'model_name': 'monte_carlo_finetune',
        }
        for term, data in sorted_terms
    ],
    'discovered_phrases': [
        {
            'phrase': ngram,
            'probability': data['probability'],
            'total_mentions': data['count'],
        }
        for ngram, data in sorted(interesting_ngrams.items(),
                                   key=lambda x: x[1]['count'], reverse=True)[:50]
    ],
}

# Save to Google Drive
timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
output_path = os.path.join(OUTPUT_DIR, f'predictions_{timestamp}.json')
latest_path = os.path.join(OUTPUT_DIR, 'predictions_latest.json')

for path in [output_path, latest_path]:
    with open(path, 'w') as f:
        json.dump(predictions_output, f, indent=2)

print(f'Predictions saved to:')
print(f'  {output_path}')
print(f'  {latest_path}')
print(f'\nDownload predictions_latest.json and place it in your bot\'s data/ directory.')
print(f'Or use the Colab API server (next cell) for live access.')

## 7. (Optional) Expose as API for Live Bot Access

Run a FastAPI server in Colab with ngrok tunnel so your local bot can call it.

In [ ]:
# Optional: Run as API server
RUN_API = False  # Set to True to expose the model as an API

if RUN_API:
    !pip install -q fastapi uvicorn pyngrok
    
    from fastapi import FastAPI
    from pydantic import BaseModel
    from pyngrok import ngrok
    import uvicorn
    import threading
    
    api = FastAPI(title='Trump Prediction API')
    
    class PredictionRequest(BaseModel):
        event_type: str = 'rally'
        location: str = 'unknown'
        audience: str = 'supporters'
        current_news: list[str] = []
        num_simulations: int = 100  # fewer for live API
        terms: list[str] = []
    
    @api.post('/predict')
    def predict(req: PredictionRequest):
        # Quick Monte Carlo run with fewer simulations
        prompt = build_generation_prompt(req.dict())
        speeches = []
        
        for _ in range(req.num_simulations):
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                output = model.generate(
                    **inputs, max_new_tokens=256,
                    temperature=0.75, do_sample=True,
                )
            speeches.append(tokenizer.decode(output[0], skip_special_tokens=True))
        
        results = {}
        terms_to_check = req.terms or tracked_terms
        for term in terms_to_check:
            containing = sum(1 for s in speeches if check_term_in_speech(s, term))
            results[term] = round(containing / len(speeches), 4)
        
        return {'predictions': results, 'num_simulations': len(speeches)}
    
    @api.get('/health')
    def health():
        return {'status': 'ok', 'model': BASE_MODEL}
    
    # Start ngrok tunnel
    # You need to set your ngrok auth token:
    # ngrok.set_auth_token('YOUR_NGROK_TOKEN')
    
    public_url = ngrok.connect(8080)
    print(f'\n{"="*60}')
    print(f'API available at: {public_url}')
    print(f'Set this in your bot\'s .env: COLAB_PREDICTION_URL={public_url}')
    print(f'{"="*60}\n')
    
    # Run in background thread
    thread = threading.Thread(
        target=uvicorn.run,
        args=(api,),
        kwargs={'host': '0.0.0.0', 'port': 8080},
        daemon=True,
    )
    thread.start()
    print('API server running in background. Keep this notebook open.')